# pydbsp tutorial

We tour the public API in six self-contained sections. The six cover Z-sets and the value-level join, pushing data through the evaluator, the doubly-incremental DLD join, sort-merge indexing, transitive closure on a graph, and Datalog. Each section is self-contained, and the same six primitives are revisited across them in successively richer configurations that the reader is invited to compare.

## 1. Z-sets and joins

A Z-set is a weighted multiset. At the value level, `join` is a predicate-projected cartesian product over two such multisets. Its predicate decides which pairs survive, while its projection decides what shape each surviving pair takes.

In [1]:
from pydbsp.zset import ZSet
from pydbsp.zset.functions.bilinear import join

employees = ZSet({(0, "kristjan"): 1, (1, "mark"): 1, (2, "mike"): 1})
salaries = ZSet({(0, "38750"): 1, (1, "50000"): 1, (2, "40000"): 1})

result = join(
    employees,
    salaries,
    lambda l, r: l[0] == r[0],
    lambda l, r: (l[0], l[1], r[1]),
)
print(result.inner)

{(0, 'kristjan', '38750'): 1, (1, 'mark', '50000'): 1, (2, 'mike', '40000'): 1}


## 2. Inputs through the Evaluator

An `Evaluator` is the stateful view over a `Circuit` together with a storage backend that the circuit reads through. `Input` operators serve as the circuit's external sources. Values are written to them with `push`, and any node at any timestamp can be read via `read`.

In [2]:
from pydbsp.circuit import Circuit
from pydbsp.compute import ComputeCtx
from pydbsp.core import Antichain, dbsp_time
from pydbsp.evaluate import Evaluator
from pydbsp.operator import Input
from pydbsp.progress import Time
from pydbsp.storage import DictStorage
from pydbsp.zset import ZSetAddition

group = ZSetAddition[tuple[int, str]]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(1)),
    group=group,
)
emp_in = Input[ZSet[tuple[int, str]]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
e.push(emp_in, employees)
print("emp_in at tick 0:", e.read(emp_in, (0,)).inner)

emp_in at tick 0: {(0, 'kristjan'): 1, (1, 'mark'): 1, (2, 'mike'): 1}


## 3. Incremental join: the 4-term DLD

The doubly-incremental bilinear join lives on a two-dimensional lattice. Both inputs arrive as one-dimensional streams that are lifted into the body via `LiftStreamIntroduction`. Unlike a pointwise join, the resulting operator expands into seventeen internal nodes. Such a node count is the price that the nested-loop derivation extracts, and an indexed variant in the next section brings it down on dense outputs.

In [3]:
from pydbsp.operator import LiftStreamIntroduction
from pydbsp.relational_operators import DeltaLiftedDeltaLiftedJoin

emp_group = ZSetAddition[tuple[int, str]]()
sal_group = ZSetAddition[tuple[int, str]]()
out_group = ZSetAddition[tuple[int, str, str]]()

e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(2)),
    group=emp_group,
)
emp_in = Input[ZSet[tuple[int, str]]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
sal_in = Input[ZSet[tuple[int, str]]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
emp_2d = LiftStreamIntroduction[ZSet[tuple[int, str]]](group=emp_group).connect(e.circuit, (emp_in,))
sal_2d = LiftStreamIntroduction[ZSet[tuple[int, str]]](group=sal_group).connect(e.circuit, (sal_in,))
incr_join = DeltaLiftedDeltaLiftedJoin[tuple[int, str], tuple[int, str], tuple[int, str, str]](
    pred=lambda l, r: l[0] == r[0],
    proj=lambda l, r: (l[0], l[1], r[1]),
    group_a=emp_group,
    group_b=sal_group,
    out_group=out_group,
).connect(e.circuit, (emp_2d, sal_2d))

e.push(emp_in, employees)
e.push(sal_in, salaries)
print("incremental join at (0, 0):", e.read(incr_join, (0, 0)).inner)

incremental join at (0, 0): {(0, 'kristjan', '38750'): 1, (1, 'mark', '50000'): 1, (2, 'mike', '40000'): 1}


## 4. Sort-merge indexed join

When both sides share a join key, the sort-merge variant wins for dense outputs. Each operand goes through `LiftIndex` first, and the indexed inputs then flow into `IndexedDeltaLiftedDeltaLiftedJoin`, which sweeps the two sorted indexes together and projects the matching pairs.

In [4]:
from pydbsp.indexed_relational_operators import (
    IndexedDeltaLiftedDeltaLiftedJoin,
    LiftIndex,
)
from pydbsp.indexed_zset import IndexedZSetAddition

e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(2)),
    group=emp_group,
)
emp_in = Input[ZSet[tuple[int, str]]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
sal_in = Input[ZSet[tuple[int, str]]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
emp_2d = LiftStreamIntroduction[ZSet[tuple[int, str]]](group=emp_group).connect(e.circuit, (emp_in,))
sal_2d = LiftStreamIntroduction[ZSet[tuple[int, str]]](group=sal_group).connect(e.circuit, (sal_in,))
emp_idx = LiftIndex[tuple[int, str], int](indexer=lambda v: v[0]).connect(e.circuit, (emp_2d,))
sal_idx = LiftIndex[tuple[int, str], int](indexer=lambda v: v[0]).connect(e.circuit, (sal_2d,))

smj = IndexedDeltaLiftedDeltaLiftedJoin[int, tuple[int, str], tuple[int, str], tuple[int, str, str]](
    proj=lambda key, l, r: (key, l[1], r[1]),
    group_a=IndexedZSetAddition[int, tuple[int, str]](emp_group, lambda v: v[0]),
    group_b=IndexedZSetAddition[int, tuple[int, str]](sal_group, lambda v: v[0]),
    out_group=out_group,
).connect(e.circuit, (emp_idx, sal_idx))

e.push(emp_in, employees)
e.push(sal_in, salaries)
print("sort-merge join at (0, 0):", e.read(smj, (0, 0)).inner)

sort-merge join at (0, 0): {(0, 'kristjan', '38750'): 1, (1, 'mark', '50000'): 1, (2, 'mike', '40000'): 1}


## 5. Reachability end-to-end

Here we walk the indexed reachability body around a three-edge chain. The inner fixpoint converges in only a handful of ticks, after which `latest(running_sum)` exposes the closure that the two-dimensional integrate has accumulated. Such a pattern matches the quickstart, with the body operator standing in for whatever recursive computation the caller wishes to express.

In [5]:
from pydbsp.indexed_relational_operators import IndexedIncrementalReachabilityBody
from pydbsp.operator import LiftIntegrate

Edge = tuple[int, int]
eg = ZSetAddition[Edge]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(2)),
    group=eg,
)
edges_1d = Input[ZSet[Edge]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
state_2d = Input[ZSet[Edge]](frontier=Antichain(dbsp_time(2))).connect(e.circuit, ())
body = IndexedIncrementalReachabilityBody[int](edge_group=eg).connect(
    e.circuit, (edges_1d, state_2d),
)
running_sum = LiftIntegrate[ZSet[Edge]](group=eg).connect(e.circuit, (body,))

e.push(edges_1d, ZSet({(0, 1): 1, (1, 2): 1, (2, 3): 1}))
for k, diff in e.saturate_inner(body, 0, is_empty=lambda d: not d.inner):
    e.push(state_2d, diff, t=(0, k))
print("TC pairs:", sorted(e.latest(running_sum).inner))

TC pairs: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 6. Datalog end-to-end

The same transitive-closure program appears here as Datalog rules and runs through `IndexedIncrementalDatalogBody`. Speed is not the point of this section, since the dedicated reach body runs faster. Generality is the point. Any first-order positive Datalog program flows through this single operator.

In [ ]:
from pydbsp.indexed_relational_operators import IndexedIncrementalDatalogBody
from pydbsp import datalog as dlg

program = ZSet({
    (("T", ("?X", "?Y")), ("E", ("?X", "?Y"))): 1,
    (("T", ("?X", "?Z")), ("E", ("?X", "?Y")), ("T", ("?Y", "?Z"))): 1,
})
edb = ZSet({("E", (0, 1)): 1, ("E", (1, 2)): 1, ("E", (2, 3)): 1})

fg = ZSetAddition[dlg.Fact]()
e = Evaluator[Time](
    circuit=Circuit[Time](),
    storage=DictStorage(),
    ctx=ComputeCtx(lattice=dbsp_time(2)),
    group=fg,
)
edb_1d = Input[ZSet[dlg.Fact]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
program_1d = Input[ZSet[dlg.Rule]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
sf = Input[ZSet[dlg.Fact]](frontier=Antichain(dbsp_time(2))).connect(e.circuit, ())
sr = Input[ZSet[dlg.ProvenanceIndexedRewrite]](frontier=Antichain(dbsp_time(2))).connect(e.circuit, ())
seed_1d = Input[ZSet[dlg.ProvenanceIndexedRewrite]](frontier=Antichain(dbsp_time(1))).connect(e.circuit, ())
program_2d = LiftStreamIntroduction[ZSet[dlg.Rule]](group=ZSetAddition[dlg.Rule]()).connect(e.circuit, (program_1d,))
body = IndexedIncrementalDatalogBody(
    fact_group=fg,
    rule_group=ZSetAddition[dlg.Rule](),
    rewrite_group=ZSetAddition[dlg.ProvenanceIndexedRewrite](),
    signal_group=ZSetAddition[dlg.Signal](),
    ext_dir_group=ZSetAddition[dlg.ExtendedDirection](),
    jorder_group=ZSetAddition[tuple[str, dlg.ColumnReference]](),
    gatekeep_group=ZSetAddition[dlg.IndexedGatekeepEntry](),
    indexed_fact_group=ZSetAddition[dlg.IndexedFact](),
).connect(e.circuit, (edb_1d, program_2d, sf, sr, seed_1d))

e.push(edb_1d, edb)
e.push(program_1d, program)
e.push(seed_1d, ZSet({(0, dlg._rewrite_monoid.identity()): 1}))

derived = fg.identity()
for k, (df, dr) in e.saturate_inner(
    body, 0, is_empty=lambda p: not p[0].inner and not p[1].inner,
):
    derived = fg.add(derived, df)
    e.push(sf, df, t=(0, k))
    e.push(sr, dr, t=(0, k))
print("Datalog derived T pairs:", sorted(k[1] for k in derived.inner if k[0] == "T"))